# M10 · Seismograms: reading, filtering, spectra, record sections

**Part 1 · week 2 · about 10 h core + 1.7 h stretch**

## 한국어 요약

이 모듈에서는 M03에서 STP로 내려받은 2019년 7월 21일 상주 지진(ML 3.9)의 파형을 ObsPy의 Stream과 Trace로 읽고, SAC 헤더의 기준 시각, b, o와 피킹 값(a, t0)을 UTC 시각으로 바꾸는 방법을 배웁니다. 표본화율과 나이퀴스트 주파수, 추세 제거와 테이퍼, 대역 통과 필터를 다루고, 영위상 필터가 초동을 앞당기고 인과 필터가 최댓값을 늦추는 현상을 직접 측정한 뒤 진폭 스펙트럼을 계산하는 함수를 작성합니다. StationMetadata(부경대학교 안병석 박사)의 계기 응답으로 카운트를 지반 속도로 바꾸고, 같은 관측소에 함께 설치된 두 계기를 비교해 그 결과를 점검합니다. S-P 시간으로 진원 거리를 구하고, 진원 거리 순으로 정렬한 레코드 섹션을 그리면서 그림에 나온 모든 숫자를 함께 출력합니다. 또 공유 서버에서 스레드 수를 세 가지 방법으로 제한해 새 Python 프로세스를 실행하고, 각각 CPU 코어를 몇 개나 쓰는지 먼저 예측한 뒤 직접 재어서 그 차이를 설명합니다.

## What you will be able to do

- Turn SAC header times (the reference time, b, o and the picks a and t0) into UTC moments, and spot header values in your own files that are missing or wrong
- Filter a seismogram with a zero-phase band-pass and compute its amplitude spectrum with the right frequency axis and amplitudes; measure why zero-phase filters move onsets earlier and causal filters delay peaks
- Remove the instrument response with StationMetadata to turn counts into ground velocity, and test the result on two instruments at one station
- Estimate hypocentral distances from S-P times and draw a record section against time after the origin, printing every number the figure shows
- Predict and measure how many CPU cores a new Python process keeps busy when its threads are limited in three different ways, and explain the differences

## Before you start

- Modules: M01, M02, M03
- Reading: Helffrich, Wookey & Bastow (2013), the chapters on SAC data files and their headers; Scherbaum (2001), the chapters on digital filters, and on causal and zero-phase filtering; Havskov & Ottemöller (2010), the chapter on signal processing (filters, spectra, instrument correction)
- Needs: stp
- Key terms (see the glossary): seismogram (지진 기록, 지진파형), seismometer (지진계), station (관측소), network code (관측망 코드), channel (채널), component (성분), trace (ObsPy Trace; 트레이스), stream (ObsPy Stream; 스트림), SAC file (SAC 파일), SAC header (SAC 헤더), reference time (SAC; 기준 시각), day of the year (Julian day; 연중 일수(줄리안 데이)), origin time (진원시, 발생 시각), UTC (협정 세계시), KST (한국 표준시), phase pick (위상 피킹, 도달 시각 읽기), sampling rate (표본화율, 샘플링 레이트), Nyquist frequency (나이퀴스트 주파수), aliasing (에일리어싱(겹침 현상)), detrend (추세 제거), taper (테이퍼), band-pass filter (대역 통과 필터), Butterworth filter (버터워스 필터), corner frequency (of a filter; 차단 주파수(코너 주파수)), filter order (corners; 필터 차수), causal filter (인과 필터), zero-phase filter (영위상 필터), group delay (군지연), in place (operation; 제자리 연산(원본 수정)), Fourier transform (푸리에 변환), spectrum (스펙트럼), decibel (dB; 데시벨), counts (카운트, 디지털 값), instrument response (계기 응답), StationXML (관측소 메타데이터 XML 형식), ground motion (지반 운동), broadband seismometer (광대역 지진계), accelerometer (가속도계), response removal (계기 응답 제거), pre-filter (사전 필터), water level (response removal; 워터 레벨), polarity reversal (극성 반전), back azimuth (역방위각), radial and transverse components (R, T; 방사 방향·접선 방향 성분), P wave (P파), S wave (S파), S-P time (S-P 시간), epicentral distance (진앙 거리), hypocentral distance (진원 거리), Vp/Vs ratio (Vp/Vs 비), velocity model (속도 구조 모델), autoregressive model (자기회귀 모형), Akaike information criterion (AIC; 아카이케 정보 기준), record section (레코드 섹션(거리별 파형 배열도)), peak ground velocity (PGV; 최대 지반 속도), Wood-Anderson seismograph (우드-앤더슨 지진계), local magnitude (ML; 국지 규모), thread (스레드), BLAS (기본 선형대수 서브프로그램), subprocess (하위 프로세스(서브프로세스)), CPU time (CPU 시간), widget (위젯)

## Passing this module

Each task is done when its `st.check(...)` cell prints PASS. The pass criteria are frozen in [criteria/M10.yaml](../../criteria/M10.yaml).

## AI use

Follow [docs/ai_policy.md](../../docs/ai_policy.md) and log every AI conversation in `work/ai_log/M10.md`.

---

© 2026 Minseong Seo (SGTL, Seoul National University) and contributors · code MIT · text CC BY 4.0 · cite CITATION.cff

In [ ]:
# Parameters: the only cell with values you may change. Run it first, then the setup cell.
DATASET = "sangju2019"                    # folder of your STP download (module M03) inside the data folder
KMA_EVENT_ID = "2019009762"               # the ML 3.9 Sangju earthquake of 21 July 2019 in the KMA catalog
BAND = "HH"                               # channel folder: HH (broadband), HG (accelerometer), EL (short period)
NEAR_STATION = "KS.SAJB"                  # network.station of the station nearest to the epicenter
FMIN_HZ, FMAX_HZ = 1.0, 10.0              # band-pass corners (Hz) for onsets, spectra and record sections
SCALE_KM = 2.0                            # record section: half-height of each trace, in km of the distance axis
VELOCITY_MODEL = "kim1983"                # shipped 1-D model; its top layer gives the speeds for S-P distances
PRE_FILT_HZ = (0.3, 0.6, 30.0, 40.0)      # response removal: frequency taper (Hz), as in M03; see Part 6
N_COLOCATED = 6                           # Part 6: number of nearest stations with HH and HG channels to compare
N_PICKED = 8                              # Part 7: number of nearest stations for S-P distances
PGV_TRACE = "KS.CHJ2..HHE"                # stretch task: the HH horizontal channel you correct
FPEAK_BAND_HZ = (1.0, 20.0)               # stretch task: band in which you look for the spectral peak
FIGURE_DIR = "figures"                    # your figure files go here, next to this notebook
RESULT_DIR = "results"                    # your result files go here, next to this notebook

In [ ]:
import sgtl_tutorial as st
S = st.init("M10")

import json
import os
import shutil
import subprocess
import sys
import textwrap
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from obspy import Stream, Trace, UTCDateTime, read, read_inventory
from obspy.geodetics import gps2dist_azimuth

Path(FIGURE_DIR).mkdir(exist_ok=True)
Path(RESULT_DIR).mkdir(exist_ok=True)

## How this notebook works

The same five steps as in the earlier modules, for every task: read the **Contract**, write a **prediction**,
replace `raise NotImplementedError(...)` under `# YOUR CODE HERE` with your code, run the check cell, and answer
the **Explain** prompt in 3-5 sentences of your own. `st.hint("M10-T1", 1)` shows a first hint; hints 2 and 3
unlock after failed checks. The checks try your functions on made-up seismograms whose answers are known from
how they were built (`st.reroll("M10-T1")` draws new ones), and on files of your own Sangju download; for
those the check computes its own answer and never prints it.

This module works on the waveforms you fetched with STP in M03. Nothing here downloads data. Seismograms take
memory (one 500-s record at 100 samples per second is 50,000 numbers), so this notebook reads only what each
part needs.

You may ask an AI tool to explain an error message or a concept. It may not write the task functions, your
predictions or your Explain answers ([docs/ai_policy.md](../../docs/ai_policy.md)); log AI conversations in
`work/ai_log/M10.md`.

## Part 0 · Your data (about 15 min)

The cell below takes the earthquake from the KMA catalog that comes with the course (times in UTC) and looks
for its waveforms in your data folder. STP named the folder of each event after its **origin time** (glossary
`origin_time`, the moment the rupture started) in UTC, written `YYYYMMDDhhmmss`. Inside it, one folder per kind
of instrument holds one SAC file per **channel** (glossary `channel`). It also reads the speeds of the top layer
of the `kim1983` **velocity model** (glossary `velocity_model`), which Part 7 needs.

In [ ]:
cat = st.data.kma_catalog()
event = cat.loc[cat["event_id"] == KMA_EVENT_ID].iloc[0]
ORIGIN = UTCDateTime(event["origin_utc"].isoformat())
EV_LAT, EV_LON = float(event["latitude"]), float(event["longitude"])
EV_DEPTH_KM, EV_ML = float(event["depth_km"]), float(event["ml"])
print(f"KMA event {KMA_EVENT_ID}: origin {ORIGIN} (UTC), {EV_LAT:.2f} N, {EV_LON:.2f} E, depth {EV_DEPTH_KM:g} km, "
      f"ML {EV_ML:.1f}")

DATA_DIR = st.data.root() / DATASET
SAC_DIR = DATA_DIR / "stp_download" / "SAC" / ORIGIN.strftime("%Y%m%d%H%M%S")
META_DIR = st.data.root() / "metadata"
if not SAC_DIR.is_dir():
    raise FileNotFoundError(f"There are no waveforms of this event in {SAC_DIR}. Fetch the {DATASET} data with STP "
                            "first (module M03), or set DATASET in the parameters cell to the folder you used.")
for band_dir in sorted(p for p in SAC_DIR.iterdir() if p.is_dir()):
    files = sorted(band_dir.glob("*.sac"))
    stations = {".".join(f.name.split(".")[1:3]) for f in files}
    print(f"{band_dir.name}: {len(files):4d} SAC files from {len(stations):3d} stations")

model = st.data.velocity_model(VELOCITY_MODEL)
VP_KM_S, VS_KM_S = float(model.iloc[0, 1]), float(model.iloc[0, 2])
print(f"{VELOCITY_MODEL}, top layer ({model.columns[1]}, {model.columns[2]}): VP {VP_KM_S} km/s, VS {VS_KM_S} km/s;",
      model.attrs["citation"].split(".")[0])

As you saw in M03, three kinds of instruments recorded this earthquake (the folder names are the first two
letters of the channel codes): **HH**, a **broadband seismometer** (glossary `broadband_seismometer`); **HG**, an
**accelerometer** (glossary `accelerometer`); **EL**, a short-period seismometer. Each record runs from 30 s
before to 470 s after the origin time, the window that M03 asked for.

## Part 1 · Seismograms in ObsPy: Stream and Trace (about 40 min)

A **seismogram** (glossary `seismogram`) is a record of ground motion against time, made by a **seismometer**
(glossary `seismometer`) at a **station** (glossary `station`) of a network (glossary `network_code`: KS is run by
KMA, KG by KIGAM). M03 explained the three letters of a channel code; the last one is the **component**
(glossary `component`), the direction of motion: Z (up), N (north) or E (east).

ObsPy (Beyreuther et al. 2010; Krischer et al. 2015) holds one channel's record in a **Trace** (glossary
`trace`): the numbers in `tr.data` (a numpy array) and their description in `tr.stats` (station, channel,
start time, sampling rate, ...). A **Stream** (glossary `stream`) is a list of Traces. `obspy.read` reads files
into a Stream; a `*` in the file name matches many files at once.

In [ ]:
near = read(str(SAC_DIR / BAND / f"*.{NEAR_STATION}.{BAND}?.sac"))
near.sort(keys=["channel"])
print(near)
z_raw = near.select(component="Z")[0]
print(z_raw.stats)
print("first five samples:", z_raw.data[:5], "| type of the numbers:", z_raw.data.dtype)

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(7.5, 5.2), sharex=True)
for ax, trace in zip(axes, near):
    ax.plot(trace.times(), trace.data, color="k", lw=0.4)
    ax.set_ylabel(f"{trace.stats.channel} (counts)")
axes[0].set_title(f"{NEAR_STATION}, raw records", fontsize=10)
axes[-1].set_xlabel("Time after the first sample (s)")
fig.tight_layout()
fig.savefig(Path(FIGURE_DIR) / "near_station_raw.png", dpi=150)
for trace in near:
    print(f"{trace.id}: {trace.stats.npts} samples at {trace.stats.sampling_rate:g} Hz, mean {trace.data.mean():.0f}, "
          f"largest |value| {np.abs(trace.data).max()} counts")

The numbers are whole numbers called **counts** (glossary `counts`): what the digitizer wrote, not yet meters
per second. Each channel also has an offset (its mean is not zero). Part 6 turns counts into ground motion.
Before that, two questions: when exactly is each sample, and what do filters do to it?

## Part 2 · SAC files: headers and time (about 40 min, then task 1)

Your files are **SAC files** (glossary `sac_file`), in the format of the Seismic Analysis Code
(Helffrich et al. 2013): a fixed **header** (glossary `sac_header`) of numbers and short texts, then the
samples. The time headers:

| header | meaning |
|---|---|
| `nzyear`, `nzjday`, `nzhour`, `nzmin`, `nzsec`, `nzmsec` | the **reference time** (glossary `reference_time`): year, day of the year, hour, minute, second, millisecond |
| `b`, `e` | time of the first and of the last sample, in seconds **after the reference time** |
| `delta`, `npts` | seconds between samples, and the number of samples |
| `o` | the origin time of the earthquake, in seconds after the reference time |
| `a`, `t0` | **phase picks** (glossary `phase_pick`): measured arrival times, here P in `a` and S in `t0`, in seconds after the reference time |

and the place headers: `stla`, `stlo`, `stel` (station latitude, longitude, elevation), `evla`, `evlo`, `evdp`
(event latitude, longitude, depth), `dist`, `az`, `baz` (distance and directions between them). Every SAC time
is a number of seconds after the one reference time. A header that is not set holds the value −12345, and
ObsPy then leaves it out of `tr.stats.sac`.

`nzjday` is the **day of the year** (glossary `day_of_year`; often called the Julian day): 1 is 1 January,
32 is 1 February, and so on up to 365 or 366. SAC times are in **UTC** (glossary `utc`), as are ObsPy's.

In [ ]:
head = z_raw.stats.sac
for key in ["nzyear", "nzjday", "nzhour", "nzmin", "nzsec", "nzmsec", "b", "e", "delta", "npts", "o", "a", "t0",
            "stla", "stlo", "stel", "evla", "evlo", "evdp", "dist", "az", "baz", "knetwk", "kstnm", "kcmpnm"]:
    print(f"{key:>7s} = {head.get(key, '(not set)')}")
print("day of the year", head.nzjday, "of", head.nzyear, "is", UTCDateTime(year=head.nzyear, julday=head.nzjday).date)
print("ObsPy's start time (first sample):", z_raw.stats.starttime)
print("KMA origin time:                  ", ORIGIN)

Read the printout slowly; three things deserve a remark.

- **The picks and the event headers are empty.** STP writes the station position, but not where the
  earthquake was or when P and S arrived. Module M20 adds picks; here they come from the catalog and from you.
- **`o` says 0.0, but the record starts about 30 s before the KMA origin time.** The reference time of these
  files is the start of the requested window, so `o = 0` would put the origin 30 s too early. A header value
  that you can check against another source (here the catalog) should be checked before you use it.
- **`b` is not 0.** The first sample sits a few milliseconds after the reference time, which is why ObsPy's
  start time does not end in `.000000`.

STP also wrote the event list it used. As in M03, its rows are in **KST** (glossary `kst`, UTC + 9 h), unlike
the SAC files:

In [ ]:
kst_rows = pd.read_csv(DATA_DIR / "event_catalog" / "event_catalog.csv")
print(kst_rows.head(3).to_string(index=False))

The second row is this earthquake: 11:04:18 KST is 02:04:18 UTC. Mixing the two clocks shifts everything by
9 hours, so convert to UTC once and stay there.

### M10-T1 · Pick times from a SAC header · core · about 60 min

**Goal:** write `pick_times_utc(sac_path)`: the UTC moments of the P pick and the S pick stored in a SAC file.

**Given:** the header table of Part 2 and ObsPy's `read`. Your own files have no picks yet; the check writes
files that have them.

**Contract:**
- Input: `sac_path`, the path of one SAC file (a `str` or a `pathlib.Path`).
- Output: a tuple `(p_utc, s_utc)`: the P pick (header `a`) and the S pick (header `t0`) as `obspy.UTCDateTime`
  moments, each `None` when that pick is not set.
- All SAC times (`b`, `e`, `o`, `a`, `t0`) are seconds after the reference time.
- Example (a made-up file): reference time 2030-05-06 07:08:09.000 (`nzyear` 2030, `nzjday` 126, `nzhour` 7,
  `nzmin` 8, `nzsec` 9, `nzmsec` 0), `b` = 0.0084, `o` = 30.0, `a` = 41.96 and no `t0` gives
  `(UTCDateTime("2030-05-06T07:08:50.960"), None)`.

**Done when** `st.check("M10-T1", func=pick_times_utc, dataset=DATASET)` passes: on 8 SAC files that the check
writes (random reference times, `b`, `o` and picks; some picks missing; a millisecond part; the last day of a
leap year), and on 10 random files of your download (copies into which the check writes picks, plus untouched
ones). The check computes the answers for your files from the raw header numbers and never prints them.

**Stuck?** `st.hint("M10-T1", 1)`

In [ ]:
# Before you code: for the example of the Contract, what would you get if you added a to ObsPy's start time, or to
# the origin time (reference time + o)? What should your function return for your own Sangju files, and why?
st.predict("M10-T1", """

""")

In [ ]:
def pick_times_utc(sac_path):
    """UTC moments of the P pick (SAC header a) and the S pick (header t0) of one SAC file.

    sac_path : str or pathlib.Path of a SAC file
    returns  : (p_utc, s_utc), each an obspy UTCDateTime, or None when that pick is not set in the header
    """
    # Step 1: read the file and find the reference time (b, the first sample, and the nz* numbers are linked).
    # Step 2: for each of the headers a and t0: None when it is missing, else the reference time plus its seconds.
    # YOUR CODE HERE
    raise NotImplementedError("M10-T1: write your code here")

In [ ]:
st.check("M10-T1", func=pick_times_utc, dataset=DATASET)

In [ ]:
near_z_file = sorted((SAC_DIR / BAND).glob(f"*.{NEAR_STATION}.{BAND}Z.sac"))[0]
print(f"your function on {near_z_file.name}:", pick_times_utc(near_z_file))

**Explain (3-5 sentences):** What is the difference between the reference time, the time of the first sample
and the origin time in your Sangju files, in seconds? Which of the three do the picks count from? What would go
wrong in a location (module M31) if someone took `o = 0.0` of these files at face value, and how did you find
out that it is wrong?

## Part 3 · Sampling, the Nyquist frequency and aliasing (about 30 min)

A digitizer measures the ground motion at regular moments: the **sampling rate** (glossary `sampling_rate`) is
the number of samples per second (100 Hz here, so `delta` = 0.01 s). A sampled record cannot represent
frequencies above the **Nyquist frequency** (glossary `nyquist_frequency`), half the sampling rate: 50 Hz here.
Worse, a wave above the Nyquist frequency does not vanish: it shows up at a false, lower frequency. This is
**aliasing** (glossary `aliasing`). Digitizers filter the signal before sampling to prevent it; you must do the
same whenever you keep fewer samples (decimate) yourself. The New Manual of Seismological Observatory Practice
(Bormann 2012) treats sampling, signals and noise in depth.

The cell below samples a 30 Hz sine at 100 Hz, then keeps every second sample (50 Hz, Nyquist 25 Hz) in two
ways: without a filter, and with ObsPy's `decimate`, which filters first. The frequency is estimated by
counting how often each record crosses zero.

In [ ]:
rate_hz, sine_hz = 100.0, 30.0
t_sine = np.arange(0.0, 4.0, 1.0 / rate_hz)
sine = Trace(np.sin(2.0 * np.pi * sine_hz * t_sine), header={"sampling_rate": rate_hz})
kept_plain = sine.copy().decimate(2, no_filter=True)     # every second sample, nothing else
kept_filtered = sine.copy().decimate(2)                  # ObsPy low-pass filters before it drops samples


def zero_crossing_hz(trace):
    """Frequency estimated from the number of sign changes per second (two per cycle)."""
    signs = np.signbit(trace.data)
    return np.count_nonzero(signs[1:] != signs[:-1]) / (trace.stats.npts * trace.stats.delta) / 2.0


for label, trace in (("100 Hz samples", sine), ("50 Hz, no filter", kept_plain), ("50 Hz, decimate", kept_filtered)):
    largest = np.abs(trace.data[20:-20]).max()           # leave out the first and last 20 samples (filter edges)
    seen = f"{zero_crossing_hz(trace):5.1f} Hz" if largest > 0.1 else "none (almost nothing left)"
    print(f"{label:>17s}: Nyquist {trace.stats.sampling_rate / 2:4.0f} Hz, largest |value| {largest:.3f}, "
          f"apparent frequency {seen}")

fig, ax = plt.subplots(figsize=(7.5, 2.6))
ax.plot(sine.times(), sine.data, ".-", color="0.6", lw=0.6, ms=3, label="30 Hz sine, sampled at 100 Hz")
ax.plot(kept_plain.times(), kept_plain.data, "s-", color="tab:red", lw=0.8, ms=3, label="every second sample")
ax.set_xlim(0.0, 0.3)
ax.set_xlabel("Time (s)")
ax.set_ylabel("Amplitude")
fig.legend(loc="upper center", ncol=2, frameon=False, fontsize=8)
fig.tight_layout(rect=(0, 0, 1, 0.84))
fig.savefig(Path(FIGURE_DIR) / "aliasing.png", dpi=150)

The red samples trace a 20 Hz wave that was never there: 30 Hz folded back around the new Nyquist frequency of
25 Hz (50 − 30 = 20). `decimate` removed the 30 Hz signal instead (its largest value is near 0), which is the
honest answer: a 50 Hz record cannot hold a 30 Hz wave.

## Part 4 · Detrend, taper, filter: zero-phase versus causal (about 60 min)

Before filtering or transforming a record, three steps are usual:

- **detrend** (glossary `detrend`): remove the mean (`"demean"`) or a straight line (`"linear"`), so the offset
  you saw in Part 1 does not leak into everything else;
- **taper** (glossary `taper`): fade both ends of the record smoothly to zero, so the filter does not see a
  sudden jump where the record starts and stops;
- **band-pass filter** (glossary `bandpass_filter`): keep the frequencies between two **corner frequencies**
  (glossary `filter_corner`) and weaken the rest. This module uses the **Butterworth filter** (glossary
  `butterworth_filter`), a common design that passes the band as evenly as possible and fades smoothly outside
  it; with more **corners** (the **filter order**, glossary `filter_order`) it cuts more steeply outside the band.

Filters come in two kinds, and the difference matters for arrival times:

- A **causal filter** (glossary `causal_filter`) uses only the present and past samples, like an instrument.
  Nothing appears before a wave arrives, but the wave comes out delayed and changed in shape: its peak moves
  later by the filter's **group delay** (glossary `group_delay`).
- A **zero-phase filter** (glossary `zero_phase_filter`) runs the causal filter forward and then backward
  (ObsPy: `zerophase=True`). The delays cancel, so peaks stay in place, but the backward run spreads energy
  **before** the arrival: a sharp onset grows small wiggles ahead of itself.

Scherbaum (2001) explains both in detail, and Havskov & Ottemöller (2010) show where they fit in routine
processing; ObsPy's Butterworth filters are designed with SciPy (Virtanen et al. 2020). The cell below filters a
made-up signal that starts sharply at 8 s.

In [ ]:
t_demo = np.arange(0.0, 20.0, 0.01)
onset_s = 8.0
burst = np.where(t_demo >= onset_s, np.exp(-(t_demo - onset_s) / 0.4) * np.sin(2.0 * np.pi * 4.0 * (t_demo - onset_s)),
                 0.0)
plain = Trace(burst, header={"delta": 0.01})
versions = {
    "unfiltered": plain,
    "causal": plain.copy().filter("bandpass", freqmin=FMIN_HZ, freqmax=FMAX_HZ, corners=4, zerophase=False),
    "zero-phase": plain.copy().filter("bandpass", freqmin=FMIN_HZ, freqmax=FMAX_HZ, corners=4, zerophase=True),
}
fig, axes = plt.subplots(3, 1, figsize=(7.5, 5.0), sharex=True)
for ax, (name, version) in zip(axes, versions.items()):
    size = np.abs(version.data)
    first = t_demo[np.argmax(size > 0.01 * size.max())]
    peak = t_demo[np.argmax(size)]
    print(f"{name:>10s}: first above 1 % of its peak at {first:.2f} s; peak at {peak:.2f} s")
    ax.plot(t_demo, version.data, color="k", lw=0.8)
    ax.axvline(onset_s, color="tab:red", lw=0.8, ls="--")
    ax.set_ylabel(name)
axes[-1].set_xlim(6.5, 10.0)
axes[-1].set_xlabel("Time (s); the red line marks the true onset")
fig.tight_layout()
fig.savefig(Path(FIGURE_DIR) / "causal_vs_zero_phase_demo.png", dpi=150)

Now the same on real data: the first seconds of the P wave at the nearest station, with the threshold set at
10 times the noise level before the earthquake. Two remarks on the code:

- ObsPy's `detrend`, `taper` and `filter` change the trace **in place** (glossary `in_place`): the trace itself
  is overwritten, and the method also returns it. Work on `tr.copy()` whenever you still need the original.
- `tr.times(reftime=ORIGIN)` gives the time of every sample in seconds after the origin time.

In [ ]:
demo = z_raw.copy()
kept = demo.data.copy()
returned = demo.detrend("demean")
print("detrend returned the same object:", returned is demo, "| its data changed:", not np.array_equal(kept, demo.data))

prepared = z_raw.copy().detrend("demean").taper(max_percentage=0.01, type="hann")
fig, ax = plt.subplots(figsize=(7.5, 3.2))
for name, zero_phase, color in (("causal", False, "tab:blue"), ("zero-phase", True, "tab:red")):
    version = prepared.copy().filter("bandpass", freqmin=FMIN_HZ, freqmax=FMAX_HZ, corners=4, zerophase=zero_phase)
    t_rel = version.times(reftime=ORIGIN)
    quiet = (t_rel > -25.0) & (t_rel < -5.0)
    noise = np.sqrt(np.mean(version.data[quiet] ** 2))
    first = t_rel[np.argmax((np.abs(version.data) > 10.0 * noise) & (t_rel > -5.0))]
    print(f"{name:>10s}: noise level {noise:7.1f} counts; first above 10 x noise at {first:5.2f} s after the origin")
    shown = (t_rel > 0.0) & (t_rel < 6.0)
    ax.plot(t_rel[shown], version.data[shown], color=color, lw=0.7, label=name)
    ax.axvline(first, color=color, lw=0.8, ls="--")
ax.set_xlabel("Time after origin (s)")
ax.set_ylabel(f"{z_raw.stats.channel} (counts)")
fig.legend(loc="upper center", ncol=2, frameon=False, fontsize=8)
fig.tight_layout(rect=(0, 0, 1, 0.88))
fig.savefig(Path(FIGURE_DIR) / "causal_vs_zero_phase_onset.png", dpi=150)

The zero-phase version crosses the threshold well before the P wave arrives: the P wave is so much larger than
the noise that even its tiny backward-spread wiggles exceed 10 times the noise. Rules of thumb: pick onsets
and polarities on causal (or unfiltered) records; measure peak amplitudes and compare waveforms on zero-phase
records.

## Part 5 · Amplitude spectra (about 30 min, then task 2)

The **Fourier transform** (glossary `fourier_transform`) writes a record as a sum of sines and cosines; its
**spectrum** (glossary `spectrum`) tells how large each frequency is. For N samples `delta` seconds apart,
numpy's `np.fft.rfft` (Harris et al. 2020) returns N // 2 + 1 complex numbers X_k, one per frequency
k / (N · delta): from 0 Hz up
to the Nyquist frequency, in steps of 1 / (N · delta), one over the record length. A longer record gives a
finer frequency step.

The size |X_k| grows with N, so it needs a scale. The **one-sided amplitude spectrum** used in this module is
2 |X_k| / N: a sine of amplitude A whose frequency is one of the k / (N · delta) then gives a peak of height A
(the 2 collects the negative frequencies that `rfft` leaves out).

Ratios of amplitudes are often given in **decibels** (glossary `decibel`): 20 log10(A1 / A2). 20 dB is a
factor of 10, 40 dB a factor of 100, and −20 dB means 10 times smaller.

### M10-T2 · Spectrum of a filtered trace · core · about 75 min

**Goal:** write `band_amplitude_spectrum(tr, fmin, fmax)`: the one-sided amplitude spectrum of a trace after a
zero-phase band-pass.

**Given:** Parts 4 and 5, and a colleague's job on the shared server. The job computes the spectra of many traces
at once as one matrix product, which NumPy hands to a **BLAS** library (glossary `blas`), the fast arithmetic
library behind NumPy. BLAS can split its work into **threads** (glossary `thread`): streams of work that run at
the same time on different CPU cores. The colleague starts the job as a **subprocess** (glossary `subprocess`:
a new program started by yours, which gets the environment that you hand it) in three ways, A, B and C, which
the provided cell below describes (run it, do not edit it; it only writes the job to a file). After your
prediction, a measurement cell runs A, B and C and reports how many cores each kept busy: its **CPU time**
(glossary `cpu_time`: the time the processor spent on it, all threads added up) divided by the wall-clock time.
You explain the three numbers at the end of this task.

**Contract:**
- Inputs: `tr`, an ObsPy Trace (evenly sampled, any sampling rate; the data may be whole numbers and may have
  an offset); `fmin`, `fmax`, the corner frequencies in Hz (0 < fmin < fmax < Nyquist frequency).
- Steps, on a copy of `tr`: remove the mean; taper 1 % at each end (Hann); band-pass with a Butterworth filter
  of 4 corners, zero-phase; then the spectrum of the whole filtered record.
- Output: a tuple `(freqs, amps)` of two 1-D numpy arrays of the same length: `freqs` in Hz from 0 to the
  Nyquist frequency in steps of 1 / (npts · delta) (the frequencies of `np.fft.rfft`), and `amps`, the one-sided
  amplitude 2 |X_k| / npts in the units of the trace.
- `tr` itself must not change.
- Example: 1000 samples at 100 Hz give 501 frequencies, 0, 0.1, ..., 50 Hz; a sine of amplitude 2 at 5 Hz,
  inside a 1-10 Hz band, gives `amps` close to 2 at 5 Hz.

**Done when** `st.check("M10-T2", func=band_amplitude_spectrum)` passes on 8 made-up traces (sampling rates from
20 to 250 Hz, odd and even numbers of samples, offsets, whole-number data): every sine inside the band within
5 % of its amplitude, every sine outside the band at least 20 dB below its amplitude.

**Stuck?** `st.hint("M10-T2", 1)`

In [ ]:
# The colleague's spectrum job. Run this cell (it writes the job to a file and runs nothing); do not edit it.
# Each run starts a new Python process with a clean environment in which the thread variables are set (without
# them the library would start one thread per core of the server), computes spectra for 5 s and prints how many
# cores it kept busy.
#   run A: starts with the thread variables at 4; the job sets them to 1 after it has imported NumPy
#   run B: starts with the thread variables at 1
#   run C: starts with the thread variables at 4; the job calls threadpoolctl's threadpool_limits(limits=1)
job_text = textwrap.dedent('''
    import os, sys, time
    import numpy as np                            # NumPy and its BLAS library are loaded here
    from threadpoolctl import threadpool_info, threadpool_limits

    n_samples, n_traces = 2048, 400               # the discrete Fourier transform written as a matrix
    k = np.arange(n_samples)
    dft = np.exp(-2j * np.pi * np.outer(k, k) / n_samples)
    traces = np.random.default_rng(1).standard_normal((n_samples, n_traces))
    test = np.abs(dft[:8, :8] @ traces[:8, :2])   # a quick test on a few samples first

    run = sys.argv[1]
    if run == "A":                                # the thread variables set to 1 from inside the job
        for name in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
            os.environ[name] = "1"
    limiter = threadpool_limits(limits=1) if run == "C" else None

    wall0, cpu0, rounds = time.perf_counter(), time.process_time(), 0
    while time.perf_counter() - wall0 < 5.0:
        spectra = np.abs(dft @ traces)
        rounds += 1
    wall, cpu = time.perf_counter() - wall0, time.process_time() - cpu0
    blas = [p["num_threads"] for p in threadpool_info() if p["user_api"] == "blas"]
    print(f"run {run}: {cpu / wall:4.2f} cores busy, {rounds * n_traces / wall:6.0f} spectra per second, "
          f"BLAS reports {blas} thread(s)")
''')
job_file = Path(RESULT_DIR) / "colleague_spectra_job.py"
job_file.write_text(job_text, encoding="utf-8")
clean_env = {"PATH": os.environ.get("PATH", ""), "HOME": os.environ.get("HOME", ""), "NUMPY_MADVISE_HUGEPAGE": "0"}
four = {name: "4" for name in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS")}
one = {name: "1" for name in four}
RUNS = {"A": four, "B": one, "C": four}            # the thread variables each run starts with
nice_prefix = ["nice", "-n", "15"] if shutil.which("nice") else []
print(f"job written to {job_file}; runs A, B and C start with",
      ", ".join(f"{run}: {start['OMP_NUM_THREADS']}" for run, start in RUNS.items()), "threads allowed")

In [ ]:
# Before you code: a trace of 3000 samples at 50 Hz holds a sine of amplitude 4 at 3 Hz and one of amplitude 4 at
# 15 Hz. With fmin = 1, fmax = 5: how many frequencies will freqs have, what is its step, and what heights do you
# expect at 3 Hz and at 15 Hz? And the colleague's job: how many cores do you expect runs A, B and C to keep busy?
st.predict("M10-T2", """

""")

Now measure: run the colleague's job in its three ways (about 20 s; each run is a new Python process under
`nice`). Compare the numbers with your prediction before you go on.

In [ ]:
for run_name, start_env in RUNS.items():
    run = subprocess.run(nice_prefix + [sys.executable, "-I", str(job_file), run_name],
                         env={**clean_env, **start_env}, capture_output=True, text=True, timeout=120)
    print(run.stdout.strip() or run.stderr.strip()[-500:])



In [ ]:
def band_amplitude_spectrum(tr, fmin, fmax):
    """One-sided amplitude spectrum of a trace after a zero-phase Butterworth band-pass (4 corners).

    tr         : obspy Trace (evenly sampled); it is not changed
    fmin, fmax : corner frequencies in Hz, 0 < fmin < fmax < Nyquist
    returns    : (freqs, amps), two 1-D numpy arrays: freqs from 0 Hz to the Nyquist frequency in steps of
                 1 / (npts * delta); amps = 2 |X_k| / npts of the demeaned, tapered (1 % Hann) and filtered data
    """
    # Step 1: prepare a copy: mean removed, ends tapered, band-pass applied (zero-phase, 4 corners).
    # Step 2: the discrete Fourier transform of the real-valued samples, and the frequency of each value.
    # Step 3: the amplitude scale of Part 5.
    # YOUR CODE HERE
    raise NotImplementedError("M10-T2: write your code here")

In [ ]:
st.check("M10-T2", func=band_amplitude_spectrum)

Use your function on the nearest station: the spectrum of 20 s of noise before the earthquake and of the
first 20 s after the origin, with a wide band (0.2-45 Hz). Where the signal rises above the noise, the record
carries information about the earthquake; elsewhere it carries mostly noise.

In [ ]:
z_noise = z_raw.slice(ORIGIN - 25.0, ORIGIN - 5.0)
z_signal = z_raw.slice(ORIGIN, ORIGIN + 20.0)
f_noise, a_noise = band_amplitude_spectrum(z_noise, 0.2, 45.0)
f_signal, a_signal = band_amplitude_spectrum(z_signal, 0.2, 45.0)
fig, ax = plt.subplots(figsize=(6.5, 4.0))
ax.loglog(f_signal[1:], a_signal[1:], color="k", lw=0.6, label="0-20 s after origin")
ax.loglog(f_noise[1:], a_noise[1:], color="0.6", lw=0.6, label="25-5 s before origin")
ax.set_xlim(0.1, 50.0)
ax.set_xlabel("Frequency (Hz)")
ax.set_ylabel(f"{z_raw.stats.channel} amplitude (counts)")
fig.legend(loc="upper center", ncol=2, frameon=False, fontsize=8)
fig.tight_layout(rect=(0, 0, 1, 0.9))
fig.savefig(Path(FIGURE_DIR) / "near_station_spectra.png", dpi=150)
for f_hz in (0.5, 1.0, 2.0, 5.0, 10.0, 20.0, 40.0):
    near_k = np.abs(f_signal - f_hz) <= 0.25 * f_hz        # average over a band around f_hz
    ratio = a_signal[near_k].mean() / a_noise[np.abs(f_noise - f_hz) <= 0.25 * f_hz].mean()
    print(f"{f_hz:5.1f} Hz: signal / noise = {ratio:8.1f} ({20.0 * np.log10(ratio):5.1f} dB)")

**Explain (3-5 sentences):** Why does your spectrum stop at the Nyquist frequency, and what sets its frequency
step? Why does the 1 % taper make the in-band peaks slightly smaller than the sines' amplitudes? In the noise
and signal spectra above, over which frequencies does the earthquake stand out, why does its margin shrink
toward low frequencies, and what makes both curves drop at the right edge? Last, the colleague's job: how many
cores did runs A, B and C keep busy, how close was your prediction, and why do the three runs differ?

## Part 6 · Counts and ground motion: testing a response removal (about 60 min)

In M03 you met the **instrument response** (glossary `instrument_response`): how the sensor and the digitizer
turn **ground motion** (glossary `ground_motion`) into counts, stage by stage, as stored in **StationXML** files
(glossary `stationxml`), and you removed it from one record (**response removal**, glossary
`response_removal`). This part looks at the responses of two kinds of sensors side by side, and then tests
response removal where it can be tested: at stations with two instruments.

The responses come from **StationMetadata**, integrated by Byeong-Seok Ahn (Pukyong National University) and
described by Ahn, Kang & Jung (2024). Cite it whenever you use it; its acknowledgment file says how:

In [ ]:
ack = META_DIR / "ACKNOWLEDGMENT.txt"
xml_dir = META_DIR / "stationxml"
if not xml_dir.is_dir():
    raise FileNotFoundError(f"No station metadata in {xml_dir}. Install it: bin/sgtl-install metadata --apply")
print("\n".join(ack.read_text(encoding="utf-8").splitlines()[:12]))

The cell below draws, for the nearest station, how many counts each sensor writes for 1 m/s of ground velocity,
frequency by frequency. A broadband seismometer measures velocity, so its curve is flat over a wide band. An
accelerometer measures acceleration, so in velocity units its curve rises in proportion to the frequency: a
ground velocity of 1 m/s at 10 Hz is a ten times larger acceleration than at 1 Hz.

In [ ]:
near_inv = read_inventory(str(xml_dir / f"{NEAR_STATION}.xml"))
freqs_hz = np.logspace(-3, np.log10(45.0), 300)
fig, ax = plt.subplots(figsize=(6.5, 4.0))
for code, color in ((f"{BAND}Z", "k"), ("HGZ", "tab:orange")):
    seed_id = f"{NEAR_STATION}..{code}"
    resp = near_inv.get_response(seed_id, ORIGIN)
    per_velocity = np.abs(resp.get_evalresp_response_for_frequencies(freqs_hz, output="VEL"))
    ax.loglog(freqs_hz, per_velocity, color=color, lw=1.0, label=code)
    print(f"{seed_id}: {per_velocity[np.argmin(np.abs(freqs_hz - 0.1))]:.4g} counts per m/s at 0.1 Hz, "
          f"{per_velocity[np.argmin(np.abs(freqs_hz - 1.0))]:.4g} at 1 Hz, "
          f"{per_velocity[np.argmin(np.abs(freqs_hz - 10.0))]:.4g} at 10 Hz")
ax.set_xlabel("Frequency (Hz)")
ax.set_ylabel("Counts per m/s of ground velocity")
fig.legend(loc="upper center", ncol=2, frameon=False, fontsize=8)
fig.tight_layout(rect=(0, 0, 1, 0.9))
fig.savefig(Path(FIGURE_DIR) / "near_station_response.png", dpi=150)

Removing the response divides the spectrum of the record by the response. M03 used the `pre_filt` safeguard
(glossary `pre_filter`): a smooth window in frequency, kept fully between the second and the third of its four
frequencies and faded to zero below the first and above the fourth (`PRE_FILT_HZ`, the values of M03). ObsPy
also offers `water_level` (glossary `water_level`): a floor, in decibels below the largest value of the
response, under which the response may not fall in the division; its default is 60 dB. Use one safeguard and
say which: here `pre_filt` with `water_level=None`. `output="VEL"` gives ground velocity in m/s ("DISP": m,
"ACC": m/s²). Some channels print the warning about sensitivities that M03 explained (a negative stage gain).

In [ ]:
vel = near.copy().detrend("demean").taper(max_percentage=0.05, type="hann")
vel.remove_response(inventory=near_inv, output="VEL", pre_filt=PRE_FILT_HZ, water_level=None)
fig, axes = plt.subplots(3, 1, figsize=(7.5, 5.2), sharex=True)
for ax, counts_tr, vel_tr in zip(axes, near, vel):
    t_rel = vel_tr.times(reftime=ORIGIN)
    ax.plot(t_rel, vel_tr.data * 1000.0, color="k", lw=0.4)
    ax.set_ylabel(f"{vel_tr.stats.channel} (mm/s)")
    print(f"{vel_tr.id}: largest |counts - mean| {np.abs(counts_tr.data - counts_tr.data.mean()).max():10.0f} -> "
          f"largest |velocity| {np.abs(vel_tr.data).max() * 1000.0:.3f} mm/s")
axes[-1].set_xlim(-5.0, 40.0)
axes[-1].set_xlabel("Time after origin (s)")
fig.tight_layout()
fig.savefig(Path(FIGURE_DIR) / "near_station_velocity.png", dpi=150)

**A test of the result.** Most KS and KG stations have a broadband seismometer (HH) and an accelerometer (HG).
After response removal both should show the same ground velocity, if they sit at the same place and their
responses are right. The cell below compares them at the `N_COLOCATED` nearest stations (band-passed
`FMIN_HZ`-`FMAX_HZ`, from 5 s before to 40 s after the origin) and lists the depth of each sensor below the
surface from the StationXML files: some sensors stand in boreholes.

In [ ]:
def velocity_record(path, inventory):
    """Ground velocity (m/s) of one SAC file: detrend, taper, response removal, band-pass, cut."""
    trace = read(str(path))[0]
    trace.detrend("demean").taper(max_percentage=0.05, type="hann")
    trace.remove_response(inventory=inventory, output="VEL", pre_filt=PRE_FILT_HZ, water_level=None)
    trace.filter("bandpass", freqmin=FMIN_HZ, freqmax=FMAX_HZ, corners=4, zerophase=True)
    return trace.trim(ORIGIN - 5.0, ORIGIN + 40.0)


def station_distance_km(path):
    """Epicentral distance (km) from the SAC header's station coordinates to the KMA epicenter."""
    sac = read(str(path), headonly=True)[0].stats.sac
    return gps2dist_azimuth(EV_LAT, EV_LON, sac.stla, sac.stlo)[0] / 1000.0


hh_z_files = sorted((SAC_DIR / "HH").glob("*HHZ.sac"))
by_distance = sorted(hh_z_files, key=station_distance_km)
rows = []
for z_file in by_distance:
    net_sta = ".".join(z_file.name.split(".")[1:3])
    if len(rows) >= 3 * N_COLOCATED or not list((SAC_DIR / "HG").glob(f"*.{net_sta}.HGZ.sac")):
        continue
    inv = read_inventory(str(xml_dir / f"{net_sta}.xml"))
    for comp in "ZNE":
        hh = velocity_record(next((SAC_DIR / "HH").glob(f"*.{net_sta}.HH{comp}.sac")), inv)
        hg = velocity_record(next((SAC_DIR / "HG").glob(f"*.{net_sta}.HG{comp}.sac")), inv)
        n = min(hh.stats.npts, hg.stats.npts)
        depth = {code: inv.select(channel=code, time=ORIGIN)[0][0][0].depth for code in ("HH" + comp, "HG" + comp)}
        rows.append({"station": net_sta, "component": comp, "epi_km": station_distance_km(z_file),
                     "HH_depth_m": depth["HH" + comp], "HG_depth_m": depth["HG" + comp],
                     "HH_peak_mm_s": 1000.0 * np.abs(hh.data).max(), "HG_peak_mm_s": 1000.0 * np.abs(hg.data).max(),
                     "HH/HG": np.abs(hh.data).max() / np.abs(hg.data).max(),
                     "correlation": np.corrcoef(hh.data[:n], hg.data[:n])[0, 1]})
colocated = pd.DataFrame(rows)
print(colocated.round({"epi_km": 1, "HH_peak_mm_s": 3, "HG_peak_mm_s": 3, "HH/HG": 2, "correlation": 3})
      .to_string(index=False))

Read the table before you go on. Where both sensors are at the same depth, the velocities should agree within
a few percent with a correlation near 1. Where they do not, ask why: different depths (the ground near the
surface amplifies waves, and a sensor in a borehole also receives the waves reflected down from the surface), a
wrong gain, or a wrong orientation. A correlation near −1 means that one record is upside down after the
removal: a **polarity reversal** (glossary `polarity_reversal`) that the metadata do not describe, or a
component that is not where its name says. A response that looks right on one instrument can still be tested
against a second one: whenever a station has two instruments, compare them before you trust its amplitudes.

**Radial and transverse.** For S waves it often helps to turn the two horizontal components toward the
earthquake: the **radial** component points from the source to the station and the **transverse** component
across (glossary `radial_transverse`). ObsPy's `rotate("NE->RT", back_azimuth=...)` needs the **back azimuth**
(glossary `back_azimuth`): the direction from the station back to the earthquake, clockwise from north.

In [ ]:
dist_m, azimuth_deg, back_azimuth_deg = gps2dist_azimuth(EV_LAT, EV_LON, z_raw.stats.sac.stla, z_raw.stats.sac.stlo)
rt = vel.copy().filter("bandpass", freqmin=FMIN_HZ, freqmax=FMAX_HZ, corners=4, zerophase=True)
rt.rotate("NE->RT", back_azimuth=back_azimuth_deg)
print(f"{NEAR_STATION}: epicentral distance {dist_m / 1000.0:.1f} km, back azimuth {back_azimuth_deg:.1f} degrees")
for trace in rt.sort(keys=["channel"]):
    print(f"{trace.id}: largest |velocity| {np.abs(trace.data).max() * 1000.0:.3f} mm/s")

**Explorer E1 (optional, not checked): a filter bench.** A **widget** (glossary `widget`) is a control in a
notebook, such as a slider or a button, that runs code again when you move it. The bench below puts the vertical
record of the nearest station under sliders: the two corner frequencies, the filter order, zero-phase or causal,
and counts or ground velocity (the response removed as above, with `PRE_FILT_HZ`). It shows the record before and
after the band-pass and, from your own `band_amplitude_spectrum` of task 2, both amplitude spectra. Your function
gets the same wide band for both records, so the two curves differ only by the filter you set. Things to try:
switch between causal and zero-phase and watch the first second of the P wave; raise the order from 2 to 8; move
the low corner below 0.5 Hz, once in counts and once in ground velocity. What changes, and why?

In [ ]:
# Optional explorer E1 (not checked): the vertical record of the nearest station on the filter bench.
st.widgets.filter_bench(z_raw, spectrum=band_amplitude_spectrum, inventory=near_inv, pre_filt=PRE_FILT_HZ,
                        fmin=FMIN_HZ, fmax=FMAX_HZ, reftime=ORIGIN, window=(-5.0, 40.0),
                        xlabel="Time after origin (s)")

## Part 7 · Distance from the S-P time (about 20 min, then task 3)

The **P wave** (glossary `p_wave`) is faster than the **S wave** (glossary `s_wave`), so at a station the S
wave arrives later, and the delay, the **S-P time** (glossary `s_minus_p_time`), grows with the distance. If
both waves travel the same straight path of length R at constant speeds VP and VS, their travel times are R / VP
and R / VS. R is the **hypocentral distance** (glossary `hypocentral_distance`): the straight-line distance
from the hypocenter to the station, while the **epicentral distance** (glossary `epicentral_distance`) runs
along the surface from the epicenter; for a focal depth h, R² = (epicentral distance)² + h². Shearer (2019)
uses S-P times the same way in his chapter on earthquake location.

### M10-T3 · Distance from an S-P time · core · about 30 min

**Goal:** write `hypo_distance_from_sp(sp_s, vp, vs)`: the hypocentral distance that gives an S-P time.

**Given:** the travel times of Part 7.

**Contract:**
- Inputs: `sp_s`, the S-P time in seconds (a number or a 1-D numpy array); `vp` and `vs`, the P and S speeds in
  km/s (numbers, `vp` > `vs`).
- Output: the hypocentral distance in km for straight paths at constant speeds: a `float` for a number, a numpy
  array of the same shape for an array.
- Example: `vp` = 6.0, `vs` = 3.5 and an S-P time of 1.0 s give 8.4 km.

**Done when** `st.check("M10-T3", func=hypo_distance_from_sp)` passes: the check draws fresh distances and
speeds, computes their S-P times from the travel times, and gives you those.

**Stuck?** `st.hint("M10-T3", 1)`

In [ ]:
# Before you code: with the top layer of kim1983 (VP 5.98, VS 3.40 km/s), how many kilometers does each second
# of S-P time stand for? Would a larger Vp/Vs ratio make that number larger or smaller?
st.predict("M10-T3", """

""")

In [ ]:
def hypo_distance_from_sp(sp_s, vp, vs):
    """Hypocentral distance (km) from an S-P time, for straight paths at constant speeds.

    sp_s   : S-P time in seconds, a number or a 1-D numpy array
    vp, vs : P and S speeds in km/s (vp > vs)
    returns: distance in km: a float for a number, a numpy array of the same shape for an array
    """
    # Step 1: write the S-P time as the difference of the two travel times of one distance.
    # Step 2: solve that for the distance, for every value at once.
    # YOUR CODE HERE
    raise NotImplementedError("M10-T3: write your code here")

In [ ]:
st.check("M10-T3", func=hypo_distance_from_sp)

Now real S-P times. Proper picking is module M20; here an automatic picker of ObsPy, `ar_pick` (run with the
example settings of the ObsPy documentation), gives rough P and S times on the three components of the
`N_PICKED` nearest stations. It fits an **autoregressive model** (glossary `autoregressive_model`: a rule that
predicts each sample from the few samples just before it) to the noise and another to the arriving wave, and
the **Akaike information criterion** (glossary `akaike_information_criterion`: a score that weighs how well a
model fits against how many numbers it needs) marks the time where switching from one model to the other
describes the record best. The table compares your S-P distances with the hypocentral distances from the
station coordinates and the KMA hypocenter.

In [ ]:
from obspy.signal.trigger import ar_pick

picked = []
for z_file in by_distance[:N_PICKED]:
    net_sta = ".".join(z_file.name.split(".")[1:3])
    three = read(str(SAC_DIR / BAND / f"*.{net_sta}.{BAND}?.sac")).trim(ORIGIN - 20.0, ORIGIN + 40.0)
    three.detrend("demean")
    z, n, e = (three.select(component=c)[0] for c in "ZNE")
    p_s, s_s = ar_pick(z.data, n.data, e.data, z.stats.sampling_rate, 1.0, 20.0, 1.0, 0.1, 4.0, 1.0, 2, 8, 0.1, 0.2)
    epi_km = station_distance_km(z_file)
    hypo_km = float(np.hypot(epi_km, EV_DEPTH_KM))
    sp_km = hypo_distance_from_sp(s_s - p_s, VP_KM_S, VS_KM_S)
    picked.append({"station": net_sta, "P_after_origin_s": z.stats.starttime + p_s - ORIGIN,
                   "S_after_origin_s": z.stats.starttime + s_s - ORIGIN, "S-P_s": s_s - p_s,
                   "hypo_km": hypo_km, "from_S-P_km": sp_km, "difference_%": 100.0 * (sp_km - hypo_km) / hypo_km})
picked = pd.DataFrame(picked)
print(picked.round(2).to_string(index=False))
print(f"median difference {picked['difference_%'].median():.1f} %")

**Explain (3-5 sentences):** Are your S-P distances longer or shorter than the distances from the catalog
hypocenter, and by how much? Name three causes that could explain it (think about the speeds and their ratio,
the picks, and the catalog location and origin time). Which of them would also shift the P times in the table
against R / VP, and which would not?

## Part 8 · Record sections (about 25 min, then task 4)

A **record section** (glossary `record_section`) draws many seismograms in one picture, each at its distance
from the source, against time after the origin. Arrivals then line up: in a plot of hypocentral distance
against time, a wave traveling straight at constant speed v lies on the line t = R / v through the origin, and
its slope gives the speed. The house rule for every figure: **print every number your figure shows**. A
reader (and you, a month later) must be able to check the figure without guessing: which traces, at which
distances, scaled by how much. Draw it with Matplotlib (Hunter 2007) as in M02.

### M10-T4 · A record section · core · about 90 min

**Goal:** write `record_section(...)`: a record section of vertical-component traces, and the table of the
numbers it shows.

**Given:** your band-pass steps from task 2, `gps2dist_azimuth` (imported in the setup cell) and Matplotlib.

**Contract:** `record_section(stream_or_paths, origin, event_lat, event_lon, event_depth_km, fmin=1.0,
fmax=10.0, scale_km=2.0)`
- Inputs: `stream_or_paths`, an ObsPy Stream, or a list of SAC file paths (`str` or `Path`) to read; every trace
  carries its station coordinates in `tr.stats.sac.stla` and `tr.stats.sac.stlo` (degrees). `origin`, the
  origin time (`UTCDateTime`); `event_lat`, `event_lon` (degrees) and `event_depth_km` (km); `fmin`, `fmax`, the
  corners of the band-pass (Hz); `scale_km`, the half-height of each trace on the distance axis.
- Processing, on copies: remove the mean, taper 1 % (Hann), band-pass zero-phase with 4 corners (as in task 2).
  `epi_km` is the epicentral distance from the epicenter to the station in km, `hypo_km` = sqrt(epi_km² +
  event_depth_km²), `peak` the largest absolute value of the filtered trace.
- Output: a tuple `(fig, table)`:
  - `fig`: a Matplotlib Figure with one Axes. Each trace is **one line** drawn with `ax.plot`: x = seconds after
    the origin (negative before it), y = `hypo_km` + `scale_km` · (filtered trace) / `peak`. The x-axis label
    says time after origin with the unit (s), the y-axis label says distance with the unit (km), and each
    trace's station code is written next to it (`ax.text`).
  - `table`: a pandas DataFrame with one row per trace, sorted by `hypo_km` (closest first), with the columns
    `id` (`tr.id`), `epi_km`, `hypo_km` and `peak`.
- The function prints the whole table, and does not change the Stream it was given.

**Done when** `st.check("M10-T4", func=record_section, dataset=DATASET)` passes on 5 made-up record sections
(6-12 traces in shuffled order, given as a Stream or as SAC files, with pulses at known times after the origin)
and on one made of 6 random HHZ files of your download (distances checked against the station coordinates in
their headers; never printed).

**Stuck?** `st.hint("M10-T4", 1)`

In [ ]:
# Before you code: in your Sangju record section, on which lines do you expect the P and the S arrivals to fall,
# and how far apart in time will they be at 50 km and at 100 km? Why must the filter be zero-phase here?
st.predict("M10-T4", """

""")

In [ ]:
def record_section(stream_or_paths, origin, event_lat, event_lon, event_depth_km, fmin=1.0, fmax=10.0,
                   scale_km=2.0):
    """Record section: each trace at its hypocentral distance, against time after the origin.

    stream_or_paths : obspy Stream, or a list of SAC paths; every trace has tr.stats.sac.stla and .stlo
    origin          : UTCDateTime of the origin
    event_lat, event_lon, event_depth_km : hypocenter (degrees, degrees, km)
    fmin, fmax      : band-pass corners (Hz), zero-phase, 4 corners
    scale_km        : half-height of each trace on the distance axis (km)
    returns         : (fig, table); table columns id, epi_km, hypo_km, peak, sorted by hypo_km; table is printed
    """
    # Step 1: one Stream of copies, whatever the input was.
    # Step 2: for each trace: filter, distances, peak.
    # Step 3: sort, draw one line per trace with labels, build and print the table.
    # YOUR CODE HERE
    raise NotImplementedError("M10-T4: write your code here")

In [ ]:
st.check("M10-T4", func=record_section, dataset=DATASET)

Now the Sangju record section of all `BAND` Z channels, with the lines t = R / VP and t = R / VS of the
`kim1983` top layer drawn on top (red and blue), saved to `FIGURE_DIR`.

In [ ]:
sangju_fig, sangju_table = record_section(sorted((SAC_DIR / BAND).glob(f"*{BAND}Z.sac")), ORIGIN, EV_LAT, EV_LON,
                                          EV_DEPTH_KM, fmin=FMIN_HZ, fmax=FMAX_HZ, scale_km=SCALE_KM)
ax = sangju_fig.axes[0]
r_line = np.array([0.0, sangju_table["hypo_km"].max() + 5.0])
for speed, color, name in ((VP_KM_S, "tab:red", "P"), (VS_KM_S, "tab:blue", "S")):
    ax.plot(r_line / speed, r_line, color=color, lw=1.0, ls="--")
    print(f"{name} line: t = R / {speed} km/s")
ax.set_xlim(-5.0, 45.0)
ax.set_title(f"Sangju ML {EV_ML:.1f}, {BAND}Z, {FMIN_HZ:g}-{FMAX_HZ:g} Hz", fontsize=10)
sangju_fig.savefig(Path(FIGURE_DIR) / "sangju_record_section.png", dpi=150)
print(f"{len(sangju_table)} traces, {sangju_table['hypo_km'].min():.1f} to {sangju_table['hypo_km'].max():.1f} km; "
      f"x from -5 to 45 s after {ORIGIN}; each trace scaled to +-{SCALE_KM} km")

**Explain (3-5 sentences):** Do the P and S arrivals follow the two straight lines? Where do they depart from
them, and what could that tell you about the speeds in the crust? Which numbers would a reader need to
reproduce your figure, and are they all in your printout? Why does each trace have its own scale, and what
does that hide?

## Stretch · How strong was the shaking?

**Peak ground velocity** (glossary `peak_ground_velocity`, PGV) is the largest absolute ground velocity of a
record. Is the PGV you measure reasonable for an earthquake of this magnitude at this distance? A simple
expectation comes from the **local magnitude** (glossary `local_magnitude`) formula that M03 used to turn
amplitudes into magnitudes; here it runs the other way. Its IASPEI standard form (Bormann & Dewey 2014), with
the constants of Hutton & Boore (1987), relates the largest amplitude A of a horizontal **Wood-Anderson
seismograph** record (glossary `wood_anderson_seismograph`; with a magnification of 1, so A is a ground
displacement) to ML and the hypocentral distance R:

$$\log_{10}(A / \mathrm{nm}) = M_\mathrm{L} - 1.11 \log_{10}(R / \mathrm{km}) - 0.00189\, (R / \mathrm{km}) + 2.09$$

The Wood-Anderson instrument records ground displacement above about 1.25 Hz. If the largest swing is a wave
of frequency f, its peak velocity is about 2πf times its peak displacement, so the expectation is
PGV ≈ 2π f A. Sheen et al. (2018) derived a distance correction for South Korea that differs from the
Californian one used here, so read the result as a factor-of-a-few check, not as a prediction.

### M10-T5 · Peak ground velocity against a magnitude expectation · stretch · about 100 min

**Goal:** measure the PGV of one HH horizontal channel of the ML 3.9 earthquake and compare it with the
expectation above.

**Given:** Part 6 (response removal), your `band_amplitude_spectrum`, the KMA event values (`EV_LAT`, `EV_LON`,
`EV_DEPTH_KM`, `EV_ML`) and `PGV_TRACE` in the parameters cell.

**Contract:**
- Read the SAC file of `PGV_TRACE` from `SAC_DIR` and its station's StationXML file from
  `META_DIR / "stationxml"`. Remove the response to ground velocity (m/s) as in Part 6 (`pre_filt=PRE_FILT_HZ`,
  `water_level=None`, after removing the mean and a 5 % taper), without a further filter. PGV is the largest
  absolute value of the whole record, in m/s.
- `hypo_km`: the hypocentral distance from the station coordinates in the header and the KMA hypocenter.
- `f_peak_hz`: the frequency of the largest value of your `band_amplitude_spectrum` of the whole velocity
  record, with the corners `FPEAK_BAND_HZ`.
- `expected_pgv_m_s`: 2π · `f_peak_hz` · A, with A from the formula above (converted from nm to m); `ratio` =
  PGV / expected.
- Write the JSON file `RESULT_DIR/pgv_check.json` with the keys `sac_path`, `stationxml`, `event_id`, `ml`,
  `hypo_km`, `pgv_m_s`, `f_peak_hz`, `expected_pgv_m_s` and `ratio` (paths as text, the rest numbers; `event_id`
  is `KMA_EVENT_ID`).
- Draw one Matplotlib figure with two panels and save it as a PNG file:
  - the velocity record from 5 s before to 40 s after the origin (time after the origin in seconds on the x
    axis), with a marker at the PGV;
  - its amplitude spectrum from your `band_amplitude_spectrum` (the one that gave `f_peak_hz`) for the
    frequencies inside `FPEAK_BAND_HZ`, with a marker at `f_peak_hz`.

  Write the unit of each panel in its y-axis label as `m/s` or `mm/s`, and write `pgv_m_s`, `f_peak_hz`,
  `expected_pgv_m_s` and `ratio` in the figure (a title, a text or a legend; at least two significant digits).
- Set `pgv_result = {"json": <path of the JSON file>, "png": <path of the PNG file>, "figure": <the Figure>,
  "spectrum": band_amplitude_spectrum}` (your function of task 2).

**Done when** `st.check("M10-T5", answer=pgv_result)` passes. The check runs your `band_amplitude_spectrum` on a
few made-up traces of task 2 again, reads your JSON file, removes the response itself (your PGV within a factor
of 1.5 of its own), recomputes the distance from the SAC header and the KMA catalog, and takes the spectral peak
of its velocity record with your own `band_amplitude_spectrum` (your `f_peak_hz` within 5 %), checks the expectation and the ratio from
your own numbers, and reads the data of your two panels: they must follow its own record and spectrum in the
units your labels name, with both markers and the four numbers in place. It never prints its own values.

**Stuck?** `st.hint("M10-T5", 1)`

In [ ]:
# Before you code: will the measured PGV be larger or smaller than the expectation, and by how much? Think about
# where the station stands and about the Korean crust.
st.predict("M10-T5", """

""")

In [ ]:
# Remove the response of PGV_TRACE, measure PGV, distance and spectral peak, compute the expectation, write the
# JSON file, draw and save the two-panel figure, and set pgv_result.
# YOUR CODE HERE
raise NotImplementedError("M10-T5: write your code here")

In [ ]:
st.check("M10-T5", answer=pgv_result)

**Explain (3-5 sentences):** By what factor does your PGV differ from the expectation? Give at least three
reasons why a single station can differ from a magnitude formula (think about the station site, the formula's
region, the frequency guess and the direction from the source). Would you trust an ML computed from this one
station, and what would you do before trusting it?

## Wrap-up

Your progress in this module (from `work/.sgtl/progress.jsonl`):

In [ ]:
st.progress("M10")

### Going further (optional, not checked)

1. Call your `record_section` on the HGZ files in `SAC_DIR / "HG"` (the accelerometers of 70 stations). Where do
   the HG traces look noisier than the HH traces, and why?
2. Draw the record section of the transverse component (rotate each station with its own back azimuth). Which
   arrival is strongest there?
3. Repeat Part 7 with a Vp/Vs ratio of 1.70 and of 1.80 (keep VP). Which ratio brings the S-P distances closest
   to the catalog distances?
4. If you also fetched the second STP dataset of `data/manifests/stp_datasets.yaml` (six small earthquakes of the
   2019 Suncheon swarm, about 23 km deep), set `DATASET` and `KMA_EVENT_ID` to one of them and make its record
   section. How do the S-P times at the nearest stations compare with Sangju?

Next: M20 picks P and S arrivals by hand and with triggers, on these same records.